# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and analyzing the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library. All dataset entities are referenced using their `@id` to ensure explicit identification and reproducibility.

### Dataset Source
The dataset is described via a Croissant schema available at:

https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()  # returns a dict
print(f"{metadata_json.get('name')}: {metadata_json.get('description')}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

Below, we programmatically retrieve the available record sets and their fields, printing their `@id`, name, and description for explicit referencing during extraction and analysis.

💡 **Tip:** All references to record sets and fields in subsequent steps will use their `@id` as required.

In [ ]:
# List all record sets with their @id, name, and available field @ids
record_sets = dataset.record_sets()
record_sets_ids = [rs["@id"] for rs in record_sets]
print("Available record sets (by @id):")
for rs in record_sets:
    print(f"- Record Set @id: {rs.get('@id')}")
    print(f"  Name: {rs.get('name')}")
    print(f"  Description: {rs.get('description')}")
    # List fields
    field_ids = [f['@id'] for f in rs.get('field', [])]
    print(f"  Fields (@id): {field_ids}")
    print("")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. All extraction is performed **explicitly by `@id`**, referencing the record set and field(s) as revealed above. 

For this notebook, let's extract the main survey record set, which typically contains participant-level data. We'll select the first record set available, but you can adjust the `main_record_set_id` variable below based on the printed overview above.

In [ ]:
dataframes = {}
main_record_set_id = record_sets_ids[0]  # Change if needed, based on printed overview
print(f"Loading records from record set: {main_record_set_id}")

# Extract all selected record sets into pandas DataFrames
for rs_id in record_sets_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Display columns for the main record set
print(f"Columns in main record set ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())

# Show a sample
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

- *Filter* records based on a numeric field using that field's `@id`.
- *Normalize* the filtered numeric field.
- *Group* data by a key (categorical) field (referenced by `@id` as per schema overview).

For demonstration, let's assume the field with `@id` `cr:GAD7_total_score` exists and is numeric, and `cr:gender` exists as a group key. Update these variable values based on the record set field overview if needed.

In [ ]:
# Specify the field @ids (change as appropriate from available field @ids above)
numeric_field_id = 'cr:GAD7_total_score'  # e.g., field measuring GAD-7 score
group_field_id = 'cr:gender'              # e.g., categorical field

# Check the presence of these fields in data
df = dataframes[main_record_set_id].copy()
assert numeric_field_id in df.columns, f"Field {numeric_field_id} not in columns!"

# Remove missing or invalid values in the numeric field
df_clean = df[df[numeric_field_id].notnull()]

# Filter records based on a threshold
threshold = 5
filtered_df = df_clean[df_clean[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} out of {len(df_clean)}")
display(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field (z-score)
filtered_df[numeric_field_id + '_normalized'] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

# Group by group_field_id, show mean and count
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
    display(grouped_df)
    count_df = filtered_df[group_field_id].value_counts().reset_index().rename(columns={group_field_id:'count', 'index': group_field_id})
    print(f"Counts by {group_field_id}:")
    display(count_df)
else:
    print(f"Group field {group_field_id} not found in columns!")

## 5. Visualization

Visualize the distribution of the selected numeric field and its relationship with the chosen group field. Customize as needed based on column presence and the field `@id`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df_clean[numeric_field_id], bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot of numeric field by group field
if group_field_id in df_clean.columns:
    plt.figure(figsize=(7,4))
    sns.boxplot(x=df_clean[group_field_id], y=df_clean[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

- We loaded a Croissant-described dataset via `mlcroissant` using only entity `@id` values for robust referencing.
- We inspected the structure and fields programmatically from schema, ensuring reproducibility and clarity.
- Simple EDA revealed the distribution and groupwise statistics of a key mental health score, demonstrating analytic workflows on AI-ready data.

Modify the field and group field `@id` above to explore further records or adapt for additional analyses. For details about available entities, always refer to the Data Overview section.